In [ ]:
import cv2
import numpy as np
import time
from pynq import Overlay
import tflite_runtime.interpreter as tflite
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# ── Load bitstream ─────────────────────────────
overlay = Overlay("/home/xilinx/jupyter_notebooks/axi/workout_classifier.bit")
print("IPs:", overlay.ip_dict.keys())
mlp = overlay.mlp_controller_0

# ── Register offsets ───────────────────────────
REG_START = 0x8C
REG_DONE  = 0x90
REG_CLASS = 0x94

class_names = {0: "Push-Up", 1: "Squat", 2: "Curl", 3: "No pose"}
class_colors_rgb = {
    0: (0, 255, 100),
    1: (0, 180, 255),
    2: (255, 100, 0),
    3: (100, 100, 100),
}

KEYPOINT_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8),
    (8, 10), (5, 11), (6, 12),
    (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16),
]

# ── Load MoveNet Lightning ─────────────────────
MODEL_PATH = "/home/xilinx/jupyter_notebooks/axi_updated_weights/movenet_lightning.tflite"
interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
INPUT_SIZE = input_details[0]['shape'][1]
print(f"MoveNet input size: {INPUT_SIZE}x{INPUT_SIZE}")

def run_movenet(frame):
    img = cv2.resize(frame, (INPUT_SIZE, INPUT_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = np.expand_dims(img, axis=0).astype(np.uint8)
    interpreter.set_tensor(input_details[0]['index'], img)
    interpreter.invoke()
    return interpreter.get_tensor(output_details[0]['index'])[0][0]

def keypoints_to_features(keypoints):
    features = []
    for kp in keypoints:
        y, x, conf = kp
        features.append(int(np.clip(x * 255, 0, 255)))
        features.append(int(np.clip(y * 255, 0, 255)))
    return features

def draw_overlay(frame, keypoints, pose, fps, conf_threshold=0.3):
    h, w = frame.shape[:2]
    color = class_colors_rgb.get(pose, (255, 255, 255))
    # convert BGR color to use in cv2
    color_bgr = (color[2], color[1], color[0])

    for p1, p2 in KEYPOINT_EDGES:
        y1, x1, c1 = keypoints[p1]
        y2, x2, c2 = keypoints[p2]
        if c1 > conf_threshold and c2 > conf_threshold:
            pt1 = (int(x1 * w), int(y1 * h))
            pt2 = (int(x2 * w), int(y2 * h))
            cv2.line(frame, pt1, pt2, color_bgr, 2)

    for kp in keypoints:
        y, x, conf = kp
        if conf > conf_threshold:
            cx, cy = int(x * w), int(y * h)
            cv2.circle(frame, (cx, cy), 5, color_bgr, -1)
            cv2.circle(frame, (cx, cy), 5, (255, 255, 255), 1)

    # banner
    banner = frame.copy()
    cv2.rectangle(banner, (0, 0), (w, 70), (20, 20, 20), -1)
    cv2.addWeighted(banner, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, class_names.get(pose, "Unknown"), (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, color_bgr, 3, cv2.LINE_AA)

    # FPS
    cv2.putText(frame, f"FPS: {fps:.1f}", (w - 120, h - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

    # keypoint confidence bar
    valid = sum(1 for kp in keypoints if kp[2] > conf_threshold)
    bar_w = int((valid / 17) * 200)
    cv2.rectangle(frame, (15, h - 40), (215, h - 20), (50, 50, 50), -1)
    cv2.rectangle(frame, (15, h - 40), (15 + bar_w, h - 20), color_bgr, -1)
    cv2.putText(frame, f"KP: {valid}/17", (220, h - 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1, cv2.LINE_AA)

    return frame

# ── Camera ─────────────────────────────────────
cap = None
for idx in [0, 1, 2]:
    cap = cv2.VideoCapture(idx, cv2.CAP_V4L2)
    if cap.isOpened():
        print(f"Camera found at index {idx}")
        break
    cap.release()

if not cap or not cap.isOpened():
    print("ERROR: No camera found")
    exit()

print("Warming up camera...")
for _ in range(10):
    cap.read()
print("Camera OK\n")

# ── Display widget ─────────────────────────────
img_widget = widgets.Image(format='jpeg', width=640, height=480)
display(img_widget)

def frame_to_jpeg(frame):
    _, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return bytes(buf)

# ── State ──────────────────────────────────────
mlp.write(REG_DONE, 0)
current_pose = 3
frame_times  = []
CONF_THRESHOLD = 0.3

# ── Main Loop ──────────────────────────────────
try:
    while True:
        loop_start = time.time()

        ret, frame = cap.read()
        if not ret:
            print("Frame read failed")
            break

        keypoints = run_movenet(frame)
        valid = sum(1 for kp in keypoints if kp[2] > CONF_THRESHOLD)

        if valid >= 10:
            features = keypoints_to_features(keypoints)
            mlp.write(REG_DONE, 0)

            for i, val in enumerate(features):
                mlp.write(0x04 + i * 4, val)

            mlp.write(REG_START, 1)
            time.sleep(0.001)
            mlp.write(REG_START, 0)

            timeout = 10000000
            count = 0
            done = 0
            while done == 0 and count < timeout:
                done = mlp.read(REG_DONE)
                count += 1

            if count < timeout:
                current_pose = mlp.read(REG_CLASS) & 0x3
            else:
                print("TIMEOUT")

        loop_time = time.time() - loop_start
        frame_times.append(loop_time)
        if len(frame_times) > 30:
            frame_times.pop(0)
        fps = 1.0 / np.mean(frame_times)

        display_frame = draw_overlay(frame.copy(), keypoints, current_pose, fps, CONF_THRESHOLD)

        # ── Update widget instead of cv2.imshow ──
        img_widget.value = frame_to_jpeg(display_frame)

except KeyboardInterrupt:
    print("\nStopped")

finally:
    cap.release()
    if frame_times:
        print(f"Average FPS: {1.0/np.mean(frame_times):.1f}")
    print("Done.")